# 05f — No respuesta de ingresos: variables, propensión de respuesta e imputación de sensibilidad

Diagnóstico, **no escribe capas ni cambia el universo ni el catálogo de features**: solo
`data/meta/no_respuesta_{inventario,propension,imputacion}_*.csv` y una fila en
`data/meta/corridas_no_respuesta.csv`. La variable objetivo **no** se imputa en el análisis principal; la
imputación de acá es solo para medir qué pasaría si se hiciera.

Variables: las de la lista de partida (vivienda `IV1–IV12`, hogar `II1–II9`, composición `IX_*`, jefe/a con
edad, educación, condición de actividad y categoría de inactividad, rama CAES, CNO abierto en carácter /
jerarquía / tecnología / calificación, `CH08`, `PP07G/H/K`), armadas por hogar en `src/no_respuesta_modelo.py`
**sin ningún monto de ingreso**.

**Train**: todo modelo se ajusta con hogares de trimestres ≤ 2022T4 (y pares con destino ≤ 2022T4). Nada del
holdout entra a ningún ajuste. LightGBM con `num_threads = 2`, validación fuera de fold agrupada por vivienda
(`CODUSU` + `AGLOMERADO`, 5 folds) para que una misma vivienda no esté en train y test.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import no_respuesta_cotas as nc
import no_respuesta_modelo as nm
import panel
import pipeline_meta
from armonizacion import trimestres_publicados
from features import CATALOGO_FEATURES, CLAVE_PAR
from trayectoria_distancia import calcular_variacion_simetrica
from imputacion_ingresos import pendiente_regresion_media, resumen_dispersion

ARMONIZADO = REPO / "data" / "01_armonizado"
DISTANCIA = REPO / "data" / "02b_distancia_linea_pobreza" / "hogar"
TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
META = REPO / "data" / "meta"
CORRIDAS = META / "corridas_no_respuesta.csv"
HOLDOUT_DESDE = (2023, 1)
ULTIMO_TRIMESTRE_TRAIN = (2022, 4)
TRIMESTRES = trimestres_publicados()
TRIMESTRES_TRAIN = [t for t in TRIMESTRES if t <= ULTIMO_TRIMESTRE_TRAIN]
CLAVE = nm.CLAVE_HOGAR_TRIMESTRE
ERAS = ("historica", "regional_2016")
N_FOLDS = 5
DECIL_BAJA_PROPENSION = 0.10
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
T0 = time.time()


def era_de(anio: int, trimestre: int) -> str:
    return "historica" if (anio, trimestre) <= (2015, 2) else "regional_2016"

## 1. Inventario de las variables candidatas

### 1a. Presencia por era (`inventario_variables.csv`)

Trimestres en que existe cada variable cruda y su tipo, por era. La rama aparece como `PP04B_COD` salvo en
2012T1–2015T2, donde INDEC la publica como `PP04B_CAES` (la armonización no la unifica: anomalía documentada;
`src/no_respuesta_modelo.py` lee la que exista).

In [ ]:
inventario = pd.read_csv(META / "inventario_variables.csv")
inventario["era"] = [era_de(a, t) for a, t in zip(inventario["anio"], inventario["trimestre"])]
CRUDAS = {"hogar": nm.VARIABLES_VIVIENDA + nm.VARIABLES_HOGAR + nm.VARIABLES_COMPOSICION,
          "individual": nm.VARIABLES_JEFE + ["CH03", "PP04D_COD", *nm.VARIABLES_RAMA]}
n_trimestres_era = inventario.groupby("era")[["anio", "trimestre"]].apply(lambda d: len(d.drop_duplicates()))
filas = []
for tipo, variables in CRUDAS.items():
    sub = inventario[(inventario["tipo"] == tipo) & inventario["variable"].isin(variables)]
    for variable in variables:
        fila = {"tipo": tipo, "variable": variable}
        for era in ERAS:
            s = sub[(sub["variable"] == variable) & (sub["era"] == era)]
            fila[f"n_trimestres_{era}"] = len(s)
            fila[f"de_{n_trimestres_era[era]}_{era}"] = n_trimestres_era[era]
            fila[f"tipo_dato_{era}"] = "/".join(sorted(s["tipo_dato"].unique()))
        filas.append(fila)
presencia = pd.DataFrame(filas)
presencia = presencia[[c for c in presencia.columns if not c.startswith("de_")]]
presencia.to_csv(META / "no_respuesta_inventario_presencia.csv", index=False)
print("trimestres por era:", n_trimestres_era.to_dict())
display(presencia.to_string(index=False))

### 1b. Tabla de hogares con las variables (todos los trimestres)

Una fila por hogar y trimestre (`nm.variables_hogar`). Solo se usa para inventario, % de NA y codificación; los
modelos de las secciones 2–3 se restringen a trimestres ≤ 2022T4.

Dos armonizaciones hechas en `variables_hogar` (anomalías encontradas en la primera corrida de este notebook,
documentadas en el reporte): `IX_MEN10` viene en NA en 2003T3–2009T2 exactamente cuando no hay menores de 10
(`IX_TOT == IX_MAYEQ10`) y pasa a 0; y el "no corresponde" de `PP04A`, `PP04C`, `PP07G1–4`, `PP07H`, `PP07I`,
`PP07K` es `0` en la histórica y vacío en la regular, y pasa a NA (las horas `PP3E_TOT` quedan NA si el jefe no
está ocupado). Sin esto, el NA funcionaría como marca de período.

In [ ]:
partes = [nm.variables_hogar(a, t, ARMONIZADO).assign(era=era_de(a, t)) for a, t in TRIMESTRES]
X_todos = pd.concat(partes, ignore_index=True)
assert not X_todos.duplicated(CLAVE).any()
print(f"{len(X_todos):,} hogares, {X_todos.shape[1]} columnas ({time.time() - T0:.0f}s)")
print("jefe/a ambiguo (0 o más de un CH03 == 1):", X_todos.groupby("era")["n_jefes"].apply(lambda s: (s != 1).mean()).round(4).to_dict())

### 1c. Codificación entre eras

Para cada variable categórica cruda (≤ 40 valores distintos), los valores que aparecen solo en una era. Un valor
exclusivo de una era es un cambio de codificación (o un código de no respuesta que cambia) que el modelo leería
como señal de época.

In [ ]:
crudas_en_x = [c for c in X_todos.columns if c in CRUDAS["hogar"] or c.removesuffix("_jefe") in CRUDAS["individual"]]
filas = []
for columna in crudas_en_x + ["rama_jefe", "cobertura_jefe", "cno_jerarquia_jefe", "cno_tecnologia_jefe", "cno_calificacion_jefe"]:
    valores = {era: set(X_todos.loc[X_todos["era"] == era, columna].dropna().unique()) for era in ERAS}
    if max(len(v) for v in valores.values()) > 40:
        filas.append({"variable": columna, "n_valores_historica": len(valores["historica"]),
                      "n_valores_regional_2016": len(valores["regional_2016"]), "solo_historica": "(continua)",
                      "solo_regional_2016": "(continua)"})
        continue
    solo_h, solo_r = valores["historica"] - valores["regional_2016"], valores["regional_2016"] - valores["historica"]
    filas.append({"variable": columna, "n_valores_historica": len(valores["historica"]),
                  "n_valores_regional_2016": len(valores["regional_2016"]),
                  "solo_historica": ",".join(map(str, sorted(solo_h))), "solo_regional_2016": ",".join(map(str, sorted(solo_r)))})
codificacion = pd.DataFrame(filas)
codificacion["cambia_codificacion"] = (codificacion["solo_historica"].ne("") | codificacion["solo_regional_2016"].ne("")) & \
    codificacion["solo_historica"].ne("(continua)")
codificacion.to_csv(META / "no_respuesta_inventario_codificacion.csv", index=False)
display(codificacion[codificacion["cambia_codificacion"]].to_string(index=False))
print("sin cambios:", ", ".join(codificacion.loc[~codificacion["cambia_codificacion"], "variable"]))

### 1d. % de NA por era y cruce con `CATALOGO_FEATURES`

`en_catalogo`: la variable ya entra (directa o como componente) en alguna feature del catálogo; se indica cuál y
si está en X. El cruce es manual, a partir de las definiciones del catálogo (`features.py`, no se edita).

In [ ]:
CRUCE_CATALOGO = {
    "IV1": ["calidad_materiales_deficitaria_t"], "IV3": ["calidad_materiales_deficitaria_t"], "IV4": ["techo_precario_t"],
    "IV6": ["agua_fuera_terreno_t"], "IV8": ["condiciones_sanitarias_deficitarias_t"],
    "IV10": ["condiciones_sanitarias_deficitarias_t"], "IV11": ["desague_a_hoyo_t"], "IV12_1": ["entorno_basural_o_inundable_t"],
    "IV12_2": ["entorno_basural_o_inundable_t"], "IV12_3": ["villa_t"], "II2": ["hacinamiento_nbi_t"],
    "II7": ["tenencia_precaria_t", "inquilino_t"], "II8": ["cocina_lena_kerosene_t"], "IX_TOT": ["n_miembros_t"],
    "CH04_jefe": ["sexo_jefe_t"], "CH06_jefe": ["edad_jefe_t"], "NIVEL_ED_jefe": ["nivel_ed_jefe_t"],
    "ESTADO_jefe": ["condicion_actividad_jefe_t"], "PP07H_jefe": ["share_precario_A_t"],
    "cno_calificacion_jefe": ["share_precario_D_t"], "n_ocupados": ["n_ocupados_t"], "n_desocupados": ["n_desocupados_t"],
    "share_ocupados": ["share_ocupados_t"],
}
en_x = {f.nombre: f.en_x for f in CATALOGO_FEATURES}
columnas_modelo = [c for c in X_todos.columns if c not in (*CLAVE, "era", "n_jefes")]
filas = []
for columna in columnas_modelo:
    destino = CRUCE_CATALOGO.get(columna, [])
    fila = {"variable": columna, "en_catalogo": bool(destino), "features_catalogo": ",".join(destino),
            "alguna_en_x": any(en_x.get(d, False) for d in destino)}
    for era in ERAS:
        fila[f"pct_na_{era}"] = 100 * X_todos.loc[X_todos["era"] == era, columna].isna().mean()
    filas.append(fila)
cruce = pd.DataFrame(filas)
cruce.to_csv(META / "no_respuesta_inventario_catalogo.csv", index=False)
display(cruce.round(1).to_string(index=False))

### 1e. Propuesta de bloque agregado para el bloque D (X base contra X ampliada)

Solo propuesta, **no se toca `features.py`**. Todas son foto en `t` (sin leakage), existen en las dos eras y no
usan montos. El % de NA viene de la tabla anterior; "no aplica" (jefe no ocupado / no asalariado) es estructural.

In [ ]:
propuesta = pd.DataFrame([
    ("rama_jefe_t", "13 grandes ramas CAES del jefe ocupado, armonizadas entre CAES EPH 2003 (≤2011T4) y CAES-MERCOSUR (≥2012T1)", "rama_jefe"),
    ("cno_caracter_jefe_t", "carácter de la ocupación (dígitos 1-2 del CNO-2001), categórica", "cno_caracter_jefe"),
    ("cno_jerarquia_jefe_t", "jerarquía (dígito 3): dirección / cuenta propia / jefatura / asalariado", "cno_jerarquia_jefe"),
    ("cno_tecnologia_jefe_t", "tecnología (dígito 4)", "cno_tecnologia_jefe"),
    ("cno_calificacion_jefe_t", "calificación (dígito 5) del jefe; hoy solo se usa agregada en share_precario_D", "cno_calificacion_jefe"),
    ("cobertura_jefe_t", "CH08 recodificada: obra social / prepaga / pública / sin cobertura", "cobertura_jefe"),
    ("n_sin_cobertura_t", "miembros con CH08 == 4", "n_sin_cobertura"),
    ("beneficios_jefe_t", "cantidad de PP07G1-4 (vacaciones, aguinaldo, enfermedad, obra social) del jefe asalariado, 0-4", "beneficios_jefe"),
    ("recibo_jefe_t", "PP07K del jefe asalariado (recibo formal / informal / factura / nada)", "PP07K_jefe"),
    ("categoria_inactividad_jefe_t", "CAT_INAC del jefe inactivo (jubilado, rentista, estudiante, ama de casa, discapacitado, otro)", "CAT_INAC_jefe"),
    ("n_jubilados_t", "miembros con CAT_INAC == 1", "n_jubilados"),
    ("ambientes_por_persona_t", "II1 / IX_TOT", "ambientes_por_persona"),
    ("tamano_establecimiento_jefe_t", "PP04C del jefe ocupado", "PP04C_jefe"),
    ("sector_jefe_t", "PP04A del jefe ocupado (estatal / privado / otro)", "PP04A_jefe"),
], columns=["feature_propuesta", "definicion", "columna_05f"])
propuesta = propuesta.merge(cruce[["variable", "pct_na_historica", "pct_na_regional_2016"]], left_on="columna_05f",
                            right_on="variable", how="left").drop(columns="variable")
propuesta = propuesta.merge(codificacion[["variable", "cambia_codificacion"]], left_on="columna_05f", right_on="variable",
                            how="left").drop(columns="variable")
propuesta.to_csv(META / "no_respuesta_inventario_propuesta_bloque_d.csv", index=False)
display(propuesta.round(1).to_string(index=False))

## 2. Propensión de respuesta por hogar

Respuesta = el hogar declara ingresos: en la regular `ingreso_no_declarado == False`; en la histórica, sin
ninguna marca de imputación en `IDIMPH`. Un modelo por era, hogares de trimestres ≤ 2022T4, sin variables de
fecha. Se compara contra la propensión de la celda aglomerado × trimestre (la del nivel
`imputacion_excluye_reponderado` vigente), también fuera de fold.

In [ ]:
partes = []
for anio, trimestre in TRIMESTRES_TRAIN:
    funcion = nc.clases_cota_historica if era_de(anio, trimestre) == "historica" else nc.clases_cota_regular
    hogares, _ = funcion(anio, trimestre, ARMONIZADO, DISTANCIA)
    partes.append(hogares[[*CLAVE, "con_no_respuesta", "clase", "ratio_cota", "pobre"]])
clases = pd.concat(partes, ignore_index=True)
datos = X_todos.merge(clases, on=CLAVE, how="inner", validate="one_to_one")
assert len(datos) == len(clases), "hogares con cota sin fila de variables"
assert (datos[["ANO4", "TRIMESTRE"]].apply(tuple, axis=1) <= ULTIMO_TRIMESTRE_TRAIN).all()
datos["declara"] = (~datos["con_no_respuesta"]).astype(int)
print(f"{len(datos):,} hogares de train ({time.time() - T0:.0f}s)")
print(datos.groupby("era")["declara"].agg(["size", "mean"]).round(4))

In [ ]:
FEATURES = [c for c in columnas_modelo if c not in ("ANO4", "TRIMESTRE")]
por_era = {era: nm.a_categoricas(datos[datos["era"] == era].reset_index(drop=True)) for era in ERAS}
filas_metricas, importancias, filas_anio = [], [], []
for era, d in por_era.items():
    fold = nm.grupos_vivienda(d, N_FOLDS)
    d["fold"] = fold
    d["p_declara"], imp = nm.predecir_fuera_de_fold(d[FEATURES], d["declara"], fold)
    celda = d.groupby(["AGLOMERADO", "ANO4", "TRIMESTRE"], observed=True)
    suma, n = celda["declara"].transform("sum"), celda["declara"].transform("size")
    suma_fold = d.groupby(["AGLOMERADO", "ANO4", "TRIMESTRE", "fold"], observed=True)["declara"].transform("sum")
    n_fold = d.groupby(["AGLOMERADO", "ANO4", "TRIMESTRE", "fold"], observed=True)["declara"].transform("size")
    d["p_celda"] = ((suma - suma_fold) / (n - n_fold).replace(0, np.nan)).fillna(d["declara"].mean())
    for modelo, columna in (("variables_hogar", "p_declara"), ("celda_aglomerado_trimestre", "p_celda")):
        filas_metricas.append({"era": era, "modelo": modelo, **nm.metricas_binarias(d["declara"], d[columna])})
    for anio, g in d.groupby("ANO4"):
        filas_anio.append({"era": era, "ANO4": anio, "auc": nm.metricas_binarias(g["declara"], g["p_declara"])["auc"],
                           "tasa_respuesta": g["declara"].mean()})
    importancias.append(imp.assign(era=era).reset_index(names="variable"))
    print(f"{era}: listo ({time.time() - T0:.0f}s)")
metricas_propension = pd.DataFrame(filas_metricas)
importancia_propension = pd.concat(importancias, ignore_index=True)
metricas_propension.to_csv(META / "no_respuesta_propension_metricas.csv", index=False)
importancia_propension.to_csv(META / "no_respuesta_propension_importancia.csv", index=False)
pd.DataFrame(filas_anio).to_csv(META / "no_respuesta_propension_auc_anio.csv", index=False)
display(metricas_propension.round(4))
display(importancia_propension.groupby("era").head(15).pivot(columns="era", values=["variable", "ganancia"]).apply(
    lambda s: s.dropna().reset_index(drop=True)).round(3))
display(pd.DataFrame(filas_anio).pivot(index="ANO4", columns="era", values="auc").round(3).T)

**Dirección de los efectos.** La ganancia dice cuánto pesa una variable, no hacia dónde. Tasa de respuesta
observada por categoría para las variables de más peso (categóricas) y por quintil (continuas), en train.

In [ ]:
filas = []
for era, d in por_era.items():
    top = importancia_propension[importancia_propension["era"] == era]["variable"].head(8)
    for variable in top:
        x = d[variable]
        grupo = x if isinstance(x.dtype, pd.CategoricalDtype) or x.nunique() <= 12 else pd.qcut(x, 5, duplicates="drop")
        tabla = d.groupby(grupo.astype("string").fillna("NA"), observed=True)["declara"].agg(["size", "mean"])
        tabla = tabla[tabla["size"] >= 500].sort_values("mean")
        for nivel, fila in pd.concat([tabla.head(3), tabla.tail(3)]).drop_duplicates().iterrows():
            filas.append({"era": era, "variable": variable, "nivel": nivel, "n": int(fila["size"]), "tasa_respuesta": fila["mean"]})
direccion = pd.DataFrame(filas)
direccion.to_csv(META / "no_respuesta_propension_direccion.csv", index=False)
display(direccion.round(3).to_string(index=False))

### 2b. Pesos inversos (recortados) y balance

Peso de cada declarante = `1 / max(p, 0,05)`, recortado al percentil 99. Tamaño efectivo de Kish relativo a `n`
(1 = sin pérdida). Balance: diferencia estandarizada de medias de cada variable numérica entre declarantes (sin
pesar y pesados) y **todos** los hogares (la población que se quiere representar). Si los pesos funcionan, la
diferencia pesada es menor.

In [ ]:
filas_pesos, filas_balance = [], []
for era, d in por_era.items():
    declarantes = d[d["declara"] == 1]
    w = nm.pesos_inversos(declarantes["p_declara"])
    d.loc[declarantes.index, "peso_propension"] = w
    filas_pesos.append({"era": era, "nivel": "hogar", **nm.resumen_pesos(w)})
    numericas = [c for c in FEATURES if not isinstance(d[c].dtype, pd.CategoricalDtype)]
    dummies = pd.get_dummies(d[["cobertura_jefe", "rama_jefe", "REGION"]].astype("string"), dummy_na=False).astype(float)
    matriz = pd.concat([d[numericas].astype(float), dummies], axis=1)
    for columna in matriz.columns:
        todos, x = matriz[columna], matriz.loc[declarantes.index, columna]
        valido = x.notna()
        desvio = todos.std()
        if not desvio or np.isnan(desvio):
            continue
        filas_balance.append({"era": era, "variable": columna,
                              "smd_sin_pesar": (x.mean() - todos.mean()) / desvio,
                              "smd_pesado": (np.average(x[valido], weights=w[valido]) - todos.mean()) / desvio})
pesos_hogar = pd.DataFrame(filas_pesos)
balance = pd.DataFrame(filas_balance)
balance.to_csv(META / "no_respuesta_propension_balance.csv", index=False)
display(pesos_hogar.round(3))
resumen_balance = balance.assign(abs_sin=balance["smd_sin_pesar"].abs(), abs_pesado=balance["smd_pesado"].abs()).groupby("era").agg(
    smd_medio_sin_pesar=("abs_sin", "mean"), smd_medio_pesado=("abs_pesado", "mean"),
    n_mayor_0_1_sin_pesar=("abs_sin", lambda s: int((s > 0.1).sum())), n_mayor_0_1_pesado=("abs_pesado", lambda s: int((s > 0.1).sum())))
display(resumen_balance.round(4))
display(balance.reindex(balance["smd_sin_pesar"].abs().sort_values(ascending=False).index).groupby("era").head(8).round(3))

### 2c. Propensión a nivel de par: insumo para `imputacion_excluye_reponderado_propension`

El experimento reponderaría **pares**, no hogares. Población: pares de train con `identidad_confirmada` que hoy son
válidos (`imputacion_excluye`) o que quedan afuera solo por ingreso (alguna punta con no respuesta). Respuesta =
par válido. X = variables del hogar en `t` (las de `t+h` no se conocen al predecir). Un modelo por era y `h`.

In [ ]:
COLUMNAS_05 = [*CLAVE_PAR, "ANO4_th", "TRIMESTRE_th", "AGLOMERADO_t", "AGLOMERADO_th", "era_t", "identidad_confirmada",
               "pobre_t", "pobre_th", "ratio_linea_pobreza_t", "ratio_linea_pobreza_th", "log_ratio_linea_pobreza_t",
               "log_ratio_linea_pobreza_th", "variacion_simetrica", "variacion_log", "par_valido_simetrica",
               "par_valido_log", "par_valido_c", "par_valido_log_c"]


def leer_pares_train(h: int) -> pd.DataFrame:
    partes = []
    for ruta in sorted((TRAYECTORIA / f"pares_h{h}").glob("ANO4=*/TRIMESTRE=*/data.parquet")):
        disponibles = pq.read_schema(ruta).names
        partes.append(pd.read_parquet(ruta, columns=[c for c in COLUMNAS_05 if c in disponibles]))
    pares = pd.concat(partes, ignore_index=True)
    pares = pares[panel.particion_train_holdout(pares, HOLDOUT_DESDE) == "train"].reset_index(drop=True)
    historica = pares["era_t"] == "historica"
    for nuevo, hist, reg in (("valido_actual", "par_valido_c", "par_valido_simetrica"),
                             ("valido_log_actual", "par_valido_log_c", "par_valido_log")):
        pares[nuevo] = np.where(historica, pares[hist].astype("boolean").fillna(False),
                                pares[reg].astype("boolean").fillna(False)).astype(bool)
    for lado in ("t", "th"):
        info = clases[[*CLAVE, "clase", "ratio_cota"]].rename(columns={
            "NRO_HOGAR": f"NRO_HOGAR_{lado}", "AGLOMERADO": f"AGLOMERADO_{lado}", "ANO4": f"ANO4_{lado}",
            "TRIMESTRE": f"TRIMESTRE_{lado}", "clase": f"clase_{lado}", "ratio_cota": f"ratio_cota_{lado}"})
        pares = pares.merge(info, on=["CODUSU", f"NRO_HOGAR_{lado}", f"AGLOMERADO_{lado}", f"ANO4_{lado}", f"TRIMESTRE_{lado}"],
                            how="left", validate="many_to_one")
    identidad = pares["identidad_confirmada"].astype("boolean").fillna(False).astype(bool)
    pares["perdido_por_ingreso"] = identidad & ~pares["valido_actual"] & (
        (pares["clase_t"] != "declarado") | (pares["clase_th"] != "declarado"))
    return pares


pares_train = {h: leer_pares_train(h) for h in (1, 4)}
X_t = X_todos[[*CLAVE, *FEATURES]].rename(columns={"NRO_HOGAR": "NRO_HOGAR_t", "AGLOMERADO": "AGLOMERADO_t",
                                                   "ANO4": "ANO4_t", "TRIMESTRE": "TRIMESTRE_t"})
FEATURES_PAR = FEATURES
filas_metricas_par, filas_pesos_par = [], []
for h, pares in pares_train.items():
    poblacion = pares[pares["valido_actual"] | pares["perdido_por_ingreso"]].merge(
        X_t, on=["CODUSU", "NRO_HOGAR_t", "AGLOMERADO_t", "ANO4_t", "TRIMESTRE_t"], how="left", validate="many_to_one")
    for era in ERAS:
        d = nm.a_categoricas(poblacion[poblacion["era_t"] == era].reset_index(drop=True))
        y = d["valido_actual"].astype(int)
        fold = nm.grupos_vivienda(d.rename(columns={"AGLOMERADO_t": "AGLOMERADO"}), N_FOLDS)
        p, _ = nm.predecir_fuera_de_fold(d[FEATURES_PAR], y, fold)
        filas_metricas_par.append({"era": era, "h": h, **nm.metricas_binarias(y, p)})
        w = nm.pesos_inversos(p[y == 1])
        filas_pesos_par.append({"era": era, "h": h, "nivel": "par", **nm.resumen_pesos(w)})
        print(f"h={h} {era}: AUC {filas_metricas_par[-1]['auc']:.3f} ({time.time() - T0:.0f}s)")
metricas_par = pd.DataFrame(filas_metricas_par)
pesos_par = pd.DataFrame(filas_pesos_par)
metricas_par.to_csv(META / "no_respuesta_propension_par_metricas.csv", index=False)
pd.concat([pesos_hogar.assign(h=pd.NA), pesos_par]).to_csv(META / "no_respuesta_propension_pesos.csv", index=False)
display(metricas_par.round(4))
display(pesos_par.round(3))

**Cómo entraría como nivel del experimento (propuesta, no implementada en `modelos.py`).**
`imputacion_excluye_reponderado_propension`: mismo train que `imputacion_excluye` (pares limpios), con peso
`1 / max(p̂, 0,05)` recortado al p99, donde `p̂` es la propensión de par limpio de esta sección **reestimada dentro
del train de cada fold** (`@solo_train`, destino ≤ (Y−1)T4), nunca con el modelo de acá, que usa todo el train.
Población de referencia por era: los pares que hoy se pierden por ingreso más los válidos (en la histórica,
coincide con la de `imputacion_excluye_reponderado`; en la regular la extiende, porque hoy los pares con no
declarante no están en ningún universo). Evaluación: siempre el conjunto limpio, como el resto de los niveles.
Criterio de coincidencia: el ya preregistrado (Spearman ρ ≥ 0,8, Jaccard top-10 ≥ 0,6, perfiles).

## 3. Imputación de sensibilidad (era regular)

Modelos sobre los declarantes de train: `pobre` (clasificación; `ITF = 0` cuenta como pobre) y `log` del ratio a
la línea (regresión, solo `ITF > 0`). Mismas variables, sin montos.

### 3a. Enmascaramiento de declarantes con perfil parecido

Predicción fuera de fold sobre los declarantes; se mira el acierto en todos y en los de **propensión de respuesta
baja** (decil inferior de `p_declara`, los que más se parecen a los no declarantes en X).

> **Advertencia.** Este acierto **no certifica nada sobre los no declarantes**: la no respuesta no es aleatoria
> (depende, plausiblemente, del propio ingreso aun condicionando en X). Que el modelo acierte en declarantes con X
> parecido dice que el modelo es bueno *para declarantes*; para los no declarantes solo hay chequeos parciales
> (3b).

In [ ]:
reg = por_era["regional_2016"]
declarantes = reg[reg["declara"] == 1].copy()
declarantes["y_pobre"] = declarantes["pobre"].astype("boolean").astype(float)
declarantes["y_log"] = np.log(declarantes["ratio_cota"].astype(float).where(declarantes["ratio_cota"] > 0))
fold_d = nm.grupos_vivienda(declarantes, N_FOLDS)
declarantes["p_pobre"], imp_pobre = nm.predecir_fuera_de_fold(declarantes[FEATURES], declarantes["y_pobre"], fold_d)
con_log = declarantes["y_log"].notna()
pred_log, imp_log = nm.predecir_fuera_de_fold(declarantes.loc[con_log, FEATURES], declarantes.loc[con_log, "y_log"],
                                              fold_d[con_log.to_numpy()], objetivo="regression")
declarantes.loc[con_log, "pred_log"] = pred_log
umbral_baja = declarantes["p_declara"].quantile(DECIL_BAJA_PROPENSION)
declarantes["grupo"] = np.where(declarantes["p_declara"] <= umbral_baja, "propension_baja_decil1", "resto")
declarantes["quintil_propension"] = pd.qcut(declarantes["p_declara"], 5, labels=[f"q{i}" for i in range(1, 6)])
tasa_aglo = declarantes.groupby("AGLOMERADO", observed=True)["y_pobre"].transform("mean")
media_log = declarantes["y_log"].mean()
filas = []
for nombre, sub in [("todos", declarantes), *declarantes.groupby("grupo"), *declarantes.groupby("quintil_propension", observed=True)]:
    fila = {"conjunto": nombre, **{f"pobre_{k}": v for k, v in nm.metricas_binarias(sub["y_pobre"], sub["p_pobre"]).items()},
            "pobre_auc_linea_base_aglomerado": nm.metricas_binarias(sub["y_pobre"], tasa_aglo[sub.index])["auc"],
            "pobre_log_loss_linea_base_aglomerado": nm.metricas_binarias(sub["y_pobre"], tasa_aglo[sub.index])["log_loss"],
            **{f"log_{k}": v for k, v in nm.metricas_continuas(sub["y_log"], sub["pred_log"]).items()},
            "log_rmse_linea_base_media": nm.metricas_continuas(sub["y_log"], pd.Series(media_log, index=sub.index))["rmse"]}
    filas.append(fila)
enmascaramiento = pd.DataFrame(filas)
enmascaramiento.to_csv(META / "no_respuesta_imputacion_enmascaramiento.csv", index=False)
pd.concat([imp_pobre.assign(modelo="pobre"), imp_log.assign(modelo="log_ratio")]).reset_index(names="variable").to_csv(
    META / "no_respuesta_imputacion_importancia.csv", index=False)
display(enmascaramiento.set_index("conjunto").T.round(4))

### 3b. Chequeos parciales sobre los no declarantes reales

Modelos reajustados con todos los declarantes de train y aplicados a los no declarantes de train. Lo único que se
sabe de ellos es la cota: un `seguro_no_pobre` **no** es pobre (su `p_pobre` debería ser baja) y un
`indeterminado` tiene ratio ≥ su cota (la imputación no debería quedar por debajo). Se reporta también la tasa de
pobreza que el modelo les asigna, por clase.

In [ ]:
modelo_pobre = nm.ajustar_lgbm(declarantes[FEATURES], declarantes["y_pobre"])
modelo_log = nm.ajustar_lgbm(declarantes.loc[con_log, FEATURES], declarantes.loc[con_log, "y_log"], objetivo="regression")
no_declarantes = reg[reg["declara"] == 0].copy()
no_declarantes["p_pobre"] = modelo_pobre.predict(no_declarantes[FEATURES])
no_declarantes["pred_log"] = modelo_log.predict(no_declarantes[FEATURES])
cota_log = np.log(no_declarantes["ratio_cota"].astype(float).where(no_declarantes["ratio_cota"] > 0))
no_declarantes["debajo_de_la_cota"] = (no_declarantes["pred_log"] < cota_log).where(cota_log.notna())
chequeo_nr = no_declarantes.groupby("clase").agg(
    n=("p_pobre", "size"), p_pobre_media=("p_pobre", "mean"), pct_p_pobre_ge_0_5=("p_pobre", lambda s: 100 * (s >= 0.5).mean()),
    pct_imputacion_debajo_cota=("debajo_de_la_cota", lambda s: 100 * s.astype(float).mean()),
    ratio_imputado_mediano=("pred_log", lambda s: float(np.exp(s.median()))))
chequeo_nr.loc["declarantes (referencia)"] = [len(declarantes), declarantes["y_pobre"].mean(), np.nan, np.nan,
                                               float(np.exp(declarantes["y_log"].median()))]
chequeo_nr.to_csv(META / "no_respuesta_imputacion_chequeo_no_declarantes.csv")
chequeo_nr.round(3)

### 3c. Prueba del 05d sobre los pares que se recuperarían

Pares regulares de train que hoy se pierden por ingreso. En cada punta sin declarar se pone el `log` del ratio
imputado (con piso en la cota cuando la cota es positiva), en dos versiones: **media condicional** (la predicción)
y **estocástica** (predicción + un residuo fuera de fold de los declarantes de propensión baja, sorteado). Se
comparan dispersión de la variación simétrica y pendiente de regresión a la media (`variacion_log` sobre
`log_ratio_t`) contra los pares limpios regulares de train, como en el 05d (allí: −0,19 sin imputar y −0,34/−0,42
con imputación INDEC en h = 1; −0,22 en regulares).

In [ ]:
rng = np.random.default_rng(0)
residuos = (declarantes["y_log"] - declarantes["pred_log"])[(declarantes["grupo"] == "propension_baja_decil1")].dropna().to_numpy()
imputacion = no_declarantes[[*CLAVE, "pred_log"]].assign(cota_log=cota_log)
imputacion["log_media"] = imputacion[["pred_log", "cota_log"]].max(axis=1)
imputacion["log_estocastica"] = np.fmax(imputacion["pred_log"] + rng.choice(residuos, len(imputacion)), imputacion["cota_log"].fillna(-np.inf))

filas = []
for h, pares in pares_train.items():
    regulares = pares[pares["era_t"] == "regional_2016"]
    limpios = regulares[regulares["valido_actual"]]
    limpios_log = regulares[regulares["valido_actual"] & regulares["valido_log_actual"]]
    filas.append({"h": h, "grupo": "limpios_regular", **resumen_dispersion(limpios["variacion_simetrica"]),
                  "pendiente_regresion_media": pendiente_regresion_media(
                      limpios_log["log_ratio_linea_pobreza_t"].astype(float), limpios_log["variacion_log"].astype(float))})
    perdidos = regulares[regulares["perdido_por_ingreso"]].copy()
    for lado in ("t", "th"):
        info = imputacion.rename(columns={"NRO_HOGAR": f"NRO_HOGAR_{lado}", "AGLOMERADO": f"AGLOMERADO_{lado}",
                                          "ANO4": f"ANO4_{lado}", "TRIMESTRE": f"TRIMESTRE_{lado}",
                                          "log_media": f"imp_media_{lado}", "log_estocastica": f"imp_estocastica_{lado}"})
        perdidos = perdidos.merge(info[["CODUSU", f"NRO_HOGAR_{lado}", f"AGLOMERADO_{lado}", f"ANO4_{lado}", f"TRIMESTRE_{lado}",
                                        f"imp_media_{lado}", f"imp_estocastica_{lado}"]],
                                  on=["CODUSU", f"NRO_HOGAR_{lado}", f"AGLOMERADO_{lado}", f"ANO4_{lado}", f"TRIMESTRE_{lado}"], how="left")
    perdidos["estado"] = np.select([(perdidos["clase_t"] != "declarado") & (perdidos["clase_th"] != "declarado"),
                                    perdidos["clase_t"] != "declarado"], ["ambas", "solo_t"], default="solo_th")
    for version in ("media", "estocastica"):
        log_t = perdidos[f"imp_{version}_t"].where(perdidos["clase_t"] != "declarado", perdidos["log_ratio_linea_pobreza_t"].astype(float))
        log_th = perdidos[f"imp_{version}_th"].where(perdidos["clase_th"] != "declarado", perdidos["log_ratio_linea_pobreza_th"].astype(float))
        ratio_t = np.exp(log_t).where(log_t.notna(), perdidos["ratio_linea_pobreza_t"].astype(float))
        ratio_th = np.exp(log_th).where(log_th.notna(), perdidos["ratio_linea_pobreza_th"].astype(float))
        perdidos["sim"] = calcular_variacion_simetrica(ratio_t, ratio_th)
        perdidos["log_var"], perdidos["log_t"] = log_th - log_t, log_t
        for estado, g in [("todos", perdidos), *perdidos.groupby("estado")]:
            filas.append({"h": h, "grupo": f"recuperados_{version}_{estado}", **resumen_dispersion(g["sim"]),
                          "pendiente_regresion_media": pendiente_regresion_media(g["log_t"], g["log_var"])})
prueba_05d = pd.DataFrame(filas)
prueba_05d.to_csv(META / "no_respuesta_imputacion_prueba_05d.csv", index=False)
display(prueba_05d.round(4).to_string(index=False))

### 3d. Era histórica: el mismo modelo contra el hot-deck de INDEC

Para los hogares históricos con imputación INDEC no hay verdad, pero sí una imputación distinta (hot-deck). Se
ajusta el modelo de `pobre` con los hogares históricos sin imputar y se compara su predicción con la condición de
pobreza que deja el ITF imputado por INDEC, en los hogares con más de la mitad del ITF imputado. Es acuerdo entre
dos imputaciones, no acierto.

In [ ]:
hist = por_era["historica"]
fraccion = pd.concat([nc.clases_cota_historica(a, t, ARMONIZADO, DISTANCIA)[0][[*CLAVE, "fraccion_max"]]
                      for a, t in TRIMESTRES_TRAIN if era_de(a, t) == "historica"], ignore_index=True)
hist = hist.merge(fraccion, on=CLAVE, how="left", validate="one_to_one")
hist["y_pobre"] = hist["pobre"].astype("boolean").astype(float)
limpios_h = hist[hist["declara"] == 1]
modelo_hist = nm.ajustar_lgbm(limpios_h[FEATURES], limpios_h["y_pobre"])
mayoria_imputada = hist[(hist["declara"] == 0) & (hist["fraccion_max"].astype(float) > 0.5)].copy()
mayoria_imputada["p_pobre"] = modelo_hist.predict(mayoria_imputada[FEATURES])
acuerdo_hotdeck = pd.DataFrame([
    {"conjunto": "imputados_fraccion_max_mayor_0_5", **nm.metricas_binarias(mayoria_imputada["y_pobre"], mayoria_imputada["p_pobre"])}])
acuerdo_hotdeck["tasa_pobreza_limpios"] = limpios_h["y_pobre"].mean()
acuerdo_hotdeck.to_csv(META / "no_respuesta_imputacion_acuerdo_hotdeck.csv", index=False)
acuerdo_hotdeck.round(4)

In [ ]:
pipeline_meta.registrar_corrida(
    CORRIDAS, "05f_no_respuesta_propension_imputacion",
    {"ULTIMO_TRIMESTRE_TRAIN": ULTIMO_TRIMESTRE_TRAIN, "N_FOLDS": N_FOLDS, "PARAMETROS_LGBM": nm.PARAMETROS_LGBM,
     "RONDAS_LGBM": nm.RONDAS_LGBM, "DECIL_BAJA_PROPENSION": DECIL_BAJA_PROPENSION},
    n_trimestres=len(TRIMESTRES_TRAIN), n_filas=0, duracion_segundos=time.time() - T0,
    commit_git=pipeline_meta.obtener_commit_git(REPO))
print(f"listo en {time.time() - T0:.0f}s")